# 03. Functional API — @entrypoint와 @task로 워크플로 만들기

## 학습 목표

Functional API의 `@entrypoint`, `@task` 패턴과 단기 메모리를 이해합니다.

## 3.1 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4.1")

In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 3.2 @task — 비동기 작업 단위

- `@task` 데코레이터로 감싸면 체크포인팅 가능
- 호출 시 즉시 Future 객체 반환, `.result()`로 대기

In [3]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver


@task
def fetch_data(url: str) -> str:
    """URL에서 데이터를 가져오는 시뮬레이션입니다."""
    return f"{url}에서 가져온 데이터"


@task
def process_data(data: str) -> str:
    """가져온 데이터를 처리합니다."""
    return f"처리 완료: {data.upper()}"


@entrypoint(checkpointer=InMemorySaver())
def data_pipeline(url: str) -> str:
    raw = fetch_data(url).result()
    processed = process_data(raw).result()
    return processed


config = {
    "configurable": {
        "thread_id": "pipe-1"
    }
}

result = data_pipeline.invoke(
    "https://example.com/api",
    {**config, **lf_config}
)

print("파이프라인 결과:", result)

파이프라인 결과: 처리 완료: HTTPS://EXAMPLE.COM/API에서 가져온 데이터


## 3.3 병렬 태스크 실행

여러 태스크를 동시에 실행합니다.

In [4]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver


@task
def analyze_sentiment(text: str) -> str:
    return f"감성({text[:20]}...): 긍정적"


@task
def extract_keywords(text: str) -> str:
    words = text.split()
    return f"키워드: {', '.join(words[:3])}"


@task
def summarize(text: str) -> str:
    return f"요약: {text[:50]}..."


@entrypoint(checkpointer=InMemorySaver())
def parallel_analysis(text: str) -> dict:
    # 3개 태스크를 병렬로 시작
    sentiment_future = analyze_sentiment(text)
    keywords_future = extract_keywords(text)
    summary_future = summarize(text)

    # 결과를 순서대로 수집
    return {
        "sentiment": sentiment_future.result(),
        "keywords": keywords_future.result(),
        "summary": summary_future.result(),
    }


config = {
    "configurable": {
        "thread_id": "parallel-1"
    }
}

result = parallel_analysis.invoke(
    "LangGraph는 상태 기반 에이전트를 구축하기 위한 강력한 프레임워크입니다",
    {**config, **lf_config}
)

for k, v in result.items():
    print(f"  {k}: {v}")

  sentiment: 감성(LangGraph는 상태 기반 에이전...): 긍정적
  keywords: 키워드: LangGraph는, 상태, 기반
  summary: 요약: LangGraph는 상태 기반 에이전트를 구축하기 위한 강력한 프레임워크입니다...


## 3.4 previous — 단기 메모리 (이전 실행 결과 접근)

In [5]:
@entrypoint(checkpointer=InMemorySaver())
def counter(increment: int, *, previous: int = 0) -> int:
    """호출할 때마다 이전 합계에 더합니다."""
    new_total = (previous or 0) + increment
    return new_total

config = {"configurable": {"thread_id": "counter-1"}}
print("호출 1:", counter.invoke(5, {**config, **lf_config}))   # 5
print("호출 2:", counter.invoke(3, {**config, **lf_config}))   # 8
print("호출 3:", counter.invoke(10, {**config, **lf_config}))  # 18

호출 1: 5
호출 2: 8
호출 3: 18


## 3.5 entrypoint.final — 반환값과 체크포인트 저장값 분리

In [6]:
@entrypoint(checkpointer=InMemorySaver())
def smart_counter(number: int, *, previous: int = 0) -> entrypoint.final[str, int]:
    """포맷된 문자열을 반환하되, 다음 호출을 위해 원시 숫자를 저장합니다."""
    new_total = (previous or 0) + number
    # value: 사용자에게 반환되는 값
    # save: 다음 호출의 previous로 저장되는 값
    return entrypoint.final(value=f"현재 합계: {new_total}", save=new_total)

config = {"configurable": {"thread_id": "smart-1"}}
print(smart_counter.invoke(10, {**config, **lf_config}))   # "현재 합계: 10"
print(smart_counter.invoke(5, {**config, **lf_config}))    # "현재 합계: 15"

현재 합계: 10
현재 합계: 15


## 3.6 결정론성 요구사항

비결정적 작업은 반드시 `@task`로 감싸야 합니다.

In [7]:
import time

# CORRECT: 비결정적 작업을 @task로 감싸기
@task
def get_timestamp() -> float:
    return time.time()


@task
def call_llm(prompt: str) -> str:
    response = model.invoke(prompt, config=lf_config)
    return response.content


@entrypoint(checkpointer=InMemorySaver())
def deterministic_workflow(prompt: str) -> dict:
    ts = get_timestamp().result()  # 체크포인트에 저장, 재실행 시 동일 값
    answer = call_llm(prompt).result()

    return {
        "timestamp": ts,
        "answer": answer
    }


config = {
    "configurable": {
        "thread_id": "det-1"
    }
}

result = deterministic_workflow.invoke(
    "한 단어로 인사해 주세요",
    {**config, **lf_config}
)

print(f"타임스탬프: {result['timestamp']}")
print(f"답변: {result['answer']}")

타임스탬프: 1772809506.6866968
답변: 안녕하세요!


## 3.7 LLM 에이전트 (Functional API)

while 루프로 ReAct 에이전트 구현

In [8]:
from langchain.tools import tool
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import add_messages

@tool
def add(a: int, b: int) -> int:
    """두 수를 더합니다."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """두 수를 곱합니다."""
    return a * b

tools = [add, multiply]
tools_by_name = {t.name: t for t in tools}
model_with_tools = model.bind_tools(tools)

@task
def call_model(messages: list) -> object:
    return model_with_tools.invoke(
        [SystemMessage(content="당신은 수학 도우미입니다.")] + messages,
        config=lf_config,
)

@task
def call_tool(tool_call: dict) -> ToolMessage:
    tool_fn = tools_by_name[tool_call["name"]]
    result = tool_fn.invoke(tool_call["args"], config=lf_config)
    return ToolMessage(content=str(result), tool_call_id=tool_call["id"])

@entrypoint(checkpointer=InMemorySaver())
def agent(messages: list) -> list:
    llm_response = call_model(messages).result()
    
    while llm_response.tool_calls:
        tool_results = [call_tool(tc).result() for tc in llm_response.tool_calls]
        messages = add_messages(messages, [llm_response] + tool_results)
        llm_response = call_model(messages).result()
    
    return add_messages(messages, llm_response)

config = {"configurable": {"thread_id": "agent-1"}}
result = agent.invoke([HumanMessage(content="7 * 8 + 3은 얼마인가요?")], {**config, **lf_config})
print("에이전트 결과:", result[-1].content)

에이전트 결과: 7 × 8 + 3의 값은 59입니다.


## 3.8 요약

| 기능 | 설명 |
|------|------|
| `@task` | 비동기 작업, 체크포인팅, 병렬 실행 |
| `@entrypoint` | 워크플로 진입점, 실행 관리 |
| `.result()` | Future 결과 동기 대기 |
| `previous` | 이전 실행 결과 접근 (단기 메모리) |
| `entrypoint.final` | 반환값 ≠ 저장값 분리 |

### 다음 단계
→ **[04_workflows.ipynb](./04_workflows.ipynb)**: 워크플로 패턴을 배웁니다.
